# Phase 2 — Data Preparation and Feature Engineering

This notebook demonstrates the reproducible pipeline for loading, merging, chronologically splitting, and engineering features from the IEEE-CIS Fraud Detection transaction and identity datasets.

## Steps:
1. Load transaction and identity datasets memory-efficiently.
2. Perform chronological train/validation/test split using `TransactionDT`.
3. Apply feature pipeline with zero data leakage.
4. Verify data integrity and target alignment.
5. Export feature metadata.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import gc
import json

# Append project root src to path
sys.path.append(os.path.abspath('..'))
print("Python interpreter path and libraries loaded.")

### 1. Load raw data and merge (left join on TransactionID)

We use the memory-efficient loading and downcasting function defined in `src/data_preparation.py`.

In [ ]:
from src.data_preparation import load_and_merge

# Loads, downcasts numeric types, and performs left join
merged_df = load_and_merge(data_dir="../data")
print(f"Merged DataFrame shape: {merged_df.shape}")

### 2. Temporal Chronological Splitting

We split the dataset into:
- 70% earliest -> Training
- 15% next -> Validation
- 15% latest -> Untouched Test

In [ ]:
from src.data_preparation import temporal_split, verify_and_assert_splits

train_df, val_df, test_df, p70, p85 = temporal_split(merged_df)
verify_and_assert_splits(train_df, val_df, test_df)

# Delete raw merged dataframe to free up memory
del merged_df
gc.collect()

### 3. Feature Pipeline & Transformation

We fit our feature pipeline on the training split only and apply it to all splits to prevent data leakage.

In [ ]:
from src.features import FeaturePipeline

pipeline = FeaturePipeline()
pipeline.fit(train_df)

# Transform each split
print("Transforming Training Split...")
X_train = pipeline.transform(train_df)

print("Transforming Validation Split...")
X_val = pipeline.transform(val_df)

print("Transforming Test Split...")
X_test = pipeline.transform(test_df)

del train_df, val_df, test_df
gc.collect()

### 4. Split Sizes & Target Distribution

Let's print the shapes and target proportions for each split to verify balance and alignment.

In [ ]:
train_rate = X_train['isFraud'].mean() * 100
val_rate = X_val['isFraud'].mean() * 100
test_rate = X_test['isFraud'].mean() * 100

print(f"Train Split shape: {X_train.shape} | Fraud Count: {X_train['isFraud'].sum():,} | Fraud Rate: {train_rate:.4f}%")
print(f"Val Split shape: {X_val.shape}   | Fraud Count: {X_val['isFraud'].sum():,} | Fraud Rate: {val_rate:.4f}%")
print(f"Test Split shape: {X_test.shape}  | Fraud Count: {X_test['isFraud'].sum():,}  | Fraud Rate: {test_rate:.4f}%")

### 5. Explicit Data Leakage Checks

We run assertions to guarantee no test records, future timestamps, or ID overlaps exist between the splits, and that `isFraud` is never in the features `X`.

In [ ]:
# Verify TransactionID uniqueness across splits
train_ids = set(X_train['TransactionID'])
val_ids = set(X_val['TransactionID'])
test_ids = set(X_test['TransactionID'])

assert len(train_ids.intersection(val_ids)) == 0, "Leakage: TransactionIDs overlap between Train and Val!"
assert len(train_ids.intersection(test_ids)) == 0, "Leakage: TransactionIDs overlap between Train and Test!"
assert len(val_ids.intersection(test_ids)) == 0, "Leakage: TransactionIDs overlap between Val and Test!"
print("Assertion Passed: Zero TransactionID overlap between splits.")

# Verify chronological ordering
assert X_train['TransactionDT'].max() <= X_val['TransactionDT'].min(), "Leakage: Temporal overlap between Train and Val!"
assert X_val['TransactionDT'].max() <= X_test['TransactionDT'].min(), "Leakage: Temporal overlap between Val and Test!"
print("Assertion Passed: Strict chronological ordering holding (no future leakage in past). ")

# Verify feature matrix excludes target, primary ID, and timestamp
feature_cols = [c for c in X_train.columns if c not in ['isFraud', 'TransactionID', 'TransactionDT']]
assert 'isFraud' not in feature_cols, "Leakage: isFraud target column exists in X features!"
assert 'TransactionID' not in feature_cols, "Leakage: TransactionID key exists in X features!"
assert 'TransactionDT' not in feature_cols, "Leakage: TransactionDT key exists in X features!"
print("Assertion Passed: isFraud target, TransactionID, and TransactionDT excluded from feature set.")

### 6. Save Feature Metadata

We serialize the feature metadata to a JSON file for model training in Phase 3.

In [ ]:
metadata = {
    "total_features": len(feature_cols),
    "feature_names": feature_cols,
    "categorical_features": [c for c in pipeline.categorical_cols if c in feature_cols],
    "numerical_features": [c for c in feature_cols if c not in pipeline.categorical_cols]
}

# Write to reports/feature_metadata.json
metadata_path = "../reports/feature_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"Successfully saved metadata for {len(feature_cols)} features to {metadata_path}")
print(f"- Categorical features: {len(metadata['categorical_features'])}")
print(f"- Numerical features: {len(metadata['numerical_features'])}")